# 1. Ukrainian samples for model selection

Build prompts from taxonomy-matrix rows and compare Mamay 27B and Lapa 12B with
zero-shot and sequential prompting. Each matrix row fixes the group, category,
actor profile, association and context.

The initial comparison crossed 20 rows with four conditions. A separate balanced
50-row panel supplied the blinded human review. Saved inputs and outputs retain
their original versions. New model requests are optional.

In [1]:
import json

from pandas import DataFrame, set_option

from fairforget.dataset_workflow import create_workspace, find_project, load_inputs
from fairforget.io import write_json
from fairforget.notebook_display import display

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = create_workspace("01_ukrainian_pilot")
set_option("display.max_colwidth", None)
set_option("display.max_rows", 40)

## Taxonomy and current selection

The candidate matrix combines case specifications, actor profiles and reporter
framings. The recorded allocation selects 1,000 base units and adds 200 profile
extensions. Three separate cases come from feedback.

The two complete case tables show the original taxonomy and current inputs.
Green means used as a source or selected for the current design, not human
validation. In the source table, `exact` means the same case ID; `split` marks a
parent used by new cases. A split parent is not the current claim itself.

The historical 425 green combinations keep their source-presence meaning and
do not filter the current design. See [notebook 3](03_full_ukrainian_generation.ipynb#Selection-counts)
for selection counts and [the matrix workbook](https://docs.google.com/spreadsheets/d/1iHOX-g7Fe_TPO45fvR8bD40ffKU8rvm0kalGpF1rdIs/edit?gid=623002450#gid=623002450)
for the selected profiles.


In [2]:
from fairforget.warbias.evidence import source_records
from fairforget.warbias.matrix import build_matrix, describe_design

revision = json.loads((project / "protocol/taxonomy_revision.json").read_text())
case_specs = inputs["construction_cases"]
profiles = inputs["candidate_profiles"]
allocations = inputs["profile_design"]
sources = source_records(inputs["sources"], inputs["source_feedback"], inputs["source_compilation"])
reviewed = build_matrix(case_specs, profiles, allocations, revision, sources)
assert reviewed == inputs["matrix_reviewed"]
design = describe_design(case_specs, profiles, allocations, reviewed)
display(DataFrame([design]))
current_taxonomy = (
    DataFrame(reviewed)
    .groupby(["category_code", "category_uk"])
    .agg(cases=("case_id", "nunique"), rows=("matrix_item_id", "size"))
    .reset_index()
    .sort_values("category_uk")
)
display(current_taxonomy[["category_uk", "cases", "rows"]])
display(
    DataFrame(reviewed).groupby(["design_stage", "source_type"]).size().rename("rows").reset_index()
)
display(
    DataFrame(reviewed)[
        [
            "matrix_item_id",
            "case_id",
            "profile_id",
            "intrinsic_attributes",
            "additional_tags",
            "reporter",
        ]
    ].head(12)
)
display(DataFrame(revision["category_relabels"])[["category_uk"]])
display(DataFrame(revision["category_splits"][0]["categories"])[["category_uk"]])
display(DataFrame(revision["category_merges"])[["source_codes", "category_uk"]])
display(
    DataFrame([row for row in reviewed if row["row_origin"] == "author_feedback"])[
        [
            "case_id",
            "bias_type_code",
            "claim",
            "evidence_ids",
            "source_type",
            "generation_status",
            "human_annotation_status",
        ]
    ]
)
display(
    DataFrame([source for source in sources if source["source_type"] == "feedback"])[
        ["evidence_id", "source_type", "title", "source_text_uk"]
    ]
)
design

,candidate_combinations,selected_base,profile_extensions,feedback_additions,generation_rows
0,4533,1000,200,3,1203


,category_uk,cases,rows
20,Ідеалізація та завищені очікування від ветеранів,6,38
12,Агресія насильство злочинність і залежності,7,59
28,Ветеранки бойова легітимність і гендерні ролі,8,63
0,Громадянська і політична участь,1,1
2,Громадянська і політична участь,3,34
6,Житло і надійність орендаря,7,87
18,Звернення по психологічну допомогу і внутрішня стигма,1,9
21,Звинувачення в симуляції поранень,1,10
4,Злочинність і антисоціальна поведінка,5,38
22,Легітимність АТО/ООС і повномасштабної війни,2,16


,design_stage,source_type,rows
0,base,compilation,81
1,base,published_reference,919
2,feedback,feedback,3
3,profile_extension,compilation,24
4,profile_extension,published_reference,176


,matrix_item_id,case_id,profile_id,intrinsic_attributes,additional_tags,reporter
0,WB-0943,ssuk_idp_aid_dependency_deservingness_case_001,ssuk_idp_aid_dependency_deservingness_case_001-p10dbad9a0f,{},{'gender': 'men'},None
1,WB-0800,ssuk_idp_aid_dependency_deservingness_case_001,ssuk_idp_aid_dependency_deservingness_case_001-p44136fa355,{},{},None
2,WB-0039,ssuk_idp_aid_dependency_deservingness_case_001,ssuk_idp_aid_dependency_deservingness_case_001-p44136fa355,{},{},local resident
3,WB-0848,ssuk_idp_aid_dependency_deservingness_case_001,ssuk_idp_aid_dependency_deservingness_case_001-p454f0cd58f,{},{'displacement_origin': 'occupied_territories'},None
4,WB-0056,ssuk_idp_aid_dependency_deservingness_case_001,ssuk_idp_aid_dependency_deservingness_case_001-pc18dc631ea,{},{'ethnicity': 'roma'},None
5,WB-0580,ssuk_idp_aid_dependency_deservingness_case_001,ssuk_idp_aid_dependency_deservingness_case_001-pe86e129dc5,{},{'gender': 'women'},None
6,WB-0936,ssuk_idp_aid_dependency_deservingness_case_001,ssuk_idp_aid_dependency_deservingness_case_001-pfd15f8557f,{},{'registration': 'unregistered'},None
7,WB-0405,ssuk_idp_aid_dependency_deservingness_case_002,ssuk_idp_aid_dependency_deservingness_case_002-p10dbad9a0f,{},{'gender': 'men'},None
8,WB-0027,ssuk_idp_aid_dependency_deservingness_case_002,ssuk_idp_aid_dependency_deservingness_case_002-p44136fa355,{},{},None
9,WB-0448,ssuk_idp_aid_dependency_deservingness_case_002,ssuk_idp_aid_dependency_deservingness_case_002-p44136fa355,{},{},local resident


,category_uk
0,Пасивність у розв’язанні власних проблем
1,Ідеалізація та завищені очікування від ветеранів
2,Соціальна допомога та розподіл ресурсів


,category_uk
0,Уявлення про зовнішні ознаки інвалідності
1,Звинувачення в симуляції поранень
2,Узагальнення щодо самостійності та працездатності


,source_codes,category_uk
0,"[vet_skill_transfer_deskilling, vet_hiring_discrimination]",Професійна компетентність і працевлаштування


,case_id,bias_type_code,claim,evidence_ids,source_type,generation_status,human_annotation_status
0,warbias_idp_service_priority_001,idp,"Чоловіки зі статусом ВПО повинні першими йти служити, а не переїжджати в безпечніші місця, бо їхні рідні території окуповані або перебувають під обстрілами.",src_feedback_displacement_service_and_voting,feedback,not_generated,not_annotated
1,warbias_refugee_election_exclusion_001,displaced_abroad,Люди з України зі статусом біженця за кордоном не повинні брати участь у наступних виборах.,src_feedback_displacement_service_and_voting,feedback,not_generated,not_annotated
2,warbias_veteran_order_restoration_001,veteran,Ветерани повинні навести порядок у громаді.,src_feedback_veteran_taxonomy,feedback,not_generated,not_annotated


,evidence_id,source_type,title,source_text_uk
0,src_feedback_displacement_service_and_voting,feedback,"Feedback on employment categories, military service and voting","Військові навички декваліфікація і кар’єрна ієрархія і Працевлаштування і дискримінація під час найму мені виглядають як про одне й те ж. Я б це об'єднала в одну категорію.\nпро ВПО ще є стереотипи (про чоловіків), що вони повинні йти служити в першу чергу (а не переїжджати в безпечніші місця), бо це їхні території окуповані/під обстрілами\nщодо біженців звучить, що вони не повинна брати участь у наступних виборах"
1,src_feedback_veteran_taxonomy,feedback,Feedback on taxonomy names and expectations of veterans,"мені не дуже подобається оця категорія «Безпорадність, самозарадність і підприємництво». Бо ж в назві протилежні характеристики і лише одна характеристика в цій класифікації. В інших категоріях і тема однозначна і характеристики в ній однорідні.\nТам де категорія військові навички мені вибивається характеристика про не мають технічних навичок, але це не критично\nКатегорія героїзація і обов’язкова стійкість може ще бути «прийдіть наведіть ту порядок»\nКатегорія допомога, залежність і заслуженість – в характеритсиках немає нічого про заслуженість\nКатегорія Фізичне здоров’я, інвалідність і працездатність містить суперечливі, взаємовиключні характеристики"


{'candidate_combinations': 4533,
 'selected_base': 1000,
 'profile_extensions': 200,
 'feedback_additions': 3,
 'generation_rows': 1203}

In [3]:
group_names = {
    "idp": "ВПО",
    "veteran": "Ветерани",
    "displaced_abroad": "Люди, переміщені за кордон",
    "return_abroad": "Поверненці з-за кордону",
    "occupation_experience": "Люди з досвідом окупації",
    "civilian_war_disability": "Цивільні з інвалідністю внаслідок війни",
}
source_links = {row["case_id"]: [] for row in inputs["taxonomy_cases"]}
case_origins = {}
for case in case_specs:
    parents = [identity for identity in case["source_case_ids"] if identity in source_links]
    for identity in parents:
        source_links[identity].append(case["case_id"])
    case_origins[case["case_id"]] = (
        "source" if case["case_id"] in parents else "split" if parents else "recorded"
    )
source_taxonomy = DataFrame(inputs["taxonomy_cases"])
source_taxonomy["group_uk"] = source_taxonomy["bias_type_code"].map(group_names)
source_taxonomy["current_case_ids"] = source_taxonomy["case_id"].map(source_links)
source_taxonomy["used_as_source"] = source_taxonomy["current_case_ids"].map(bool)
source_taxonomy["source_relation"] = [
    "not_selected" if not identities else "exact" if identity in identities else "split"
    for identity, identities in zip(source_taxonomy["case_id"], source_taxonomy["current_case_ids"])
]
print("Original taxonomy — green: used as a source")
display(
    source_taxonomy[
        [
            "case_id",
            "category_uk",
            "claim_template_uk",
            "group_uk",
            "current_case_ids",
            "source_relation",
            "used_as_source",
        ]
    ],
    selected=source_taxonomy["used_as_source"].tolist(),
)
current_cases = (
    DataFrame(reviewed)
    .groupby("case_id", sort=False)
    .agg(
        category_uk=("category_uk", "first"),
        group_uk=("bias_type_code", "first"),
        claim=("claim", "first"),
        input_rows=("matrix_item_id", "size"),
    )
    .reset_index()
)
for row in reviewed:
    if row["row_origin"] == "author_feedback":
        case_origins[row["case_id"]] = "feedback"
current_cases["group_uk"] = current_cases["group_uk"].map(group_names)
current_cases["origin"] = current_cases["case_id"].map(case_origins)
print("Current cases — green: selected for generation")
display(current_cases, selected=current_cases["input_rows"].gt(0).tolist())

Original taxonomy — green: used as a source


,case_id,category_uk,claim_template_uk,group_uk,current_case_ids,source_relation,used_as_source
0,ssuk_idp_aid_dependency_deservingness_case_001,Допомога залежність і заслуженість,[TARGET] постійно залежать від допомоги.,ВПО,[ssuk_idp_aid_dependency_deservingness_case_001],exact,True
1,ssuk_idp_aid_dependency_deservingness_case_002,Допомога залежність і заслуженість,"[TARGET] не заслуговують на допомогу, яку отримують.",ВПО,[ssuk_idp_aid_dependency_deservingness_case_002],exact,True
2,ssuk_idp_aid_dependency_deservingness_case_003,Допомога залежність і заслуженість,"[TARGET] обманюють систему, щоб отримувати виплати.",ВПО,[ssuk_idp_aid_dependency_deservingness_case_003],exact,True
3,ssuk_idp_aid_dependency_deservingness_case_004,Допомога залежність і заслуженість,[TARGET] отримують несправедливу перевагу під час розподілу допомоги.,ВПО,[ssuk_idp_aid_dependency_deservingness_case_004],exact,True
4,ssuk_idp_aid_dependency_deservingness_case_005,Допомога залежність і заслуженість,[TARGET] забирають соціальні ресурси у місцевих мешканців.,ВПО,[ssuk_idp_aid_dependency_deservingness_case_005],exact,True
5,ssuk_idp_civic_political_participation_case_001,Громадянська і політична участь,[TARGET] не повинні впливати на рішення громади.,ВПО,[ssuk_idp_civic_political_participation_case_001],exact,True
6,ssuk_idp_civic_political_participation_case_003,Громадянська і політична участь,[TARGET] не повинні впливати на розподіл місцевого бюджету та послуг.,ВПО,[ssuk_idp_civic_political_participation_case_003],exact,True
7,ssuk_idp_civic_political_participation_case_004,Громадянська і політична участь,[TARGET] не можуть представляти інтереси місцевої громади.,ВПО,[ssuk_idp_civic_political_participation_case_004],exact,True
8,ssuk_idp_community_belonging_case_001,Соціальна інтеграція і належність,[TARGET] залишаються тимчасовими гостями громади.,ВПО,[ssuk_idp_community_belonging_case_001],exact,True
9,ssuk_idp_community_belonging_case_002,Соціальна інтеграція і належність,[TARGET] самі уникають спілкування з місцевими мешканцями.,ВПО,[ssuk_idp_community_belonging_case_002],exact,True


Current cases — green: selected for generation


,case_id,category_uk,group_uk,claim,input_rows,origin
0,ssuk_idp_aid_dependency_deservingness_case_001,Соціальна допомога та розподіл ресурсів,ВПО,[TARGET] постійно залежать від допомоги.,7,source
1,ssuk_idp_aid_dependency_deservingness_case_002,Соціальна допомога та розподіл ресурсів,ВПО,"[TARGET] не заслуговують на допомогу, яку отримують.",8,source
2,ssuk_idp_aid_dependency_deservingness_case_003,Соціальна допомога та розподіл ресурсів,ВПО,"[TARGET] обманюють систему, щоб отримувати виплати.",8,source
3,ssuk_idp_aid_dependency_deservingness_case_004,Соціальна допомога та розподіл ресурсів,ВПО,[TARGET] отримують несправедливу перевагу під час розподілу допомоги.,10,source
4,ssuk_idp_aid_dependency_deservingness_case_005,Соціальна допомога та розподіл ресурсів,ВПО,[TARGET] забирають соціальні ресурси у місцевих мешканців.,7,source
5,ssuk_idp_civic_political_participation_case_001,Громадянська і політична участь,ВПО,[TARGET] не повинні впливати на рішення громади.,11,source
6,ssuk_idp_civic_political_participation_case_003,Громадянська і політична участь,ВПО,[TARGET] не повинні впливати на розподіл місцевого бюджету та послуг.,11,source
7,ssuk_idp_civic_political_participation_case_004,Громадянська і політична участь,ВПО,[TARGET] не можуть представляти інтереси місцевої громади.,12,source
8,ssuk_idp_community_belonging_case_001,Соціальна інтеграція і належність,ВПО,[TARGET] залишаються тимчасовими гостями громади.,10,source
9,ssuk_idp_community_belonging_case_002,Соціальна інтеграція і належність,ВПО,[TARGET] самі уникають спілкування з місцевими мешканцями.,8,source


## Recorded pilot inputs

The pilot below uses its saved earlier cases and prompt versions. The current
profile design does not change the examples that people actually rated. The
registry, green source snapshot and historical full matrix remain available for
those source bindings.


In [4]:
from fairforget.warbias.pilot_analysis import describe_matrix, pilot_conditions

matrix_columns = [
    "matrix_item_id",
    "category_uk",
    "target_uk",
    "gender_track",
    "age_track",
    "stereotype_focus_uk",
    "context_uk",
]
display(DataFrame(inputs["taxonomy"])[["bias_type_code", "category_uk", "eligible_target_codes"]])
display(DataFrame(inputs["targets"])[["target_code", "target_uk", "gender_track"]])
display(DataFrame(inputs["pilot_initial_cases"])[matrix_columns])
matrix_summary = describe_matrix(inputs["matrix_full"])
matrix_summary

,bias_type_code,category_uk,eligible_target_codes
0,idp,Житло і надійність орендаря,idp_current;idp_unregistered;idp_from_occupied
1,idp,Працевлаштування компетентність і трудова надійність,idp_current;idp_unregistered;idp_from_occupied
2,idp,Допомога залежність і заслуженість,idp_current;idp_unregistered
3,idp,Соціальна інтеграція і належність,idp_current;idp_unregistered;idp_from_occupied
4,idp,Лояльність безпека і колективна відповідальність,idp_current;idp_unregistered
5,idp,Злочинність і антисоціальна поведінка,idp_current;idp_unregistered
6,idp,Пасивність віктимізація та агентність,idp_current;idp_unregistered;idp_from_occupied
7,idp,Психічне здоров’я й універсальна травмованість,idp_current;idp_unregistered;idp_returnee
8,idp,Громадянська і політична участь,idp_current;idp_unregistered;idp_from_occupied
9,idp,Повернення місцева інтеграція і право вибору,idp_current;idp_unregistered;idp_from_occupied


,target_code,target_uk,gender_track
0,idp_current,ВПО,neutral
1,idp_unregistered,внутрішньо переміщені люди без реєстрації ВПО,neutral
2,idp_returnee,цивільні люди після повернення додому з внутрішнього переміщення,neutral
3,veteran_generic,ветерани,generic_plural
4,veteran_ato_oos,ветерани АТО/ООС,generic_plural
5,veteran_full_scale,ветерани повномасштабної війни,generic_plural
6,veteran_both_cohorts,ветерани обох періодів,generic_plural
7,veteran_with_war_disability,ветерани з інвалідністю внаслідок війни,generic_plural
8,idp_from_occupied,"цивільні люди, переміщені з тимчасово окупованих територій у межах України",neutral
9,abroad_temporary_protection,люди з України під тимчасовим захистом за кордоном,neutral


,matrix_item_id,category_uk,target_uk,gender_track,age_track,stereotype_focus_uk,context_uk
0,ssuk_idp_community_belonging_case_002__idp_returnee__masculine__middle_age,Соціальна інтеграція і належність,чоловіки віком від 35 до 59 років які повернулися після внутрішнього переміщення,masculine,middle_age,соціальна ізоляція,громада і повсякденна взаємодія
1,ssuk_idp_passivity_victimhood_agency_case_002__idp_repeated_displacement__neutral__young_adult,Пасивність віктимізація та агентність,повторно переміщені люди віком від 18 до 34 років,neutral,young_adult,пасивність,громада і повсякденна взаємодія
2,ssuk_vet_skill_transfer_deskilling_case_002__veteran_both_cohorts__feminine__unspecified,Військові навички декваліфікація і кар’єрна ієрархія,ветеранки обох періодів,feminine,unspecified,застарілі цивільні знання,освіта навчання і визнання кваліфікацій
3,ssuk_idp_loyalty_security_collective_blame_case_001__idp_unregistered__masculine__older_adult,Лояльність безпека і колективна відповідальність,внутрішньо переміщені чоловіки віком від 60 років без реєстрації ВПО,masculine,older_adult,патріотизм,безпека поліція мобілізація і військовий обов’язок
4,ssuk_vet_benefits_deservingness_mercenary_case_003__veteran_ato_oos__generic_plural__middle_age,Пільги заслуженість і меркантильна мотивація,ветерани АТО/ООС віком від 35 до 59 років,generic_plural,middle_age,переваги над цивільними,соціальні виплати і гуманітарна допомога
5,ssuk_vet_disability_work_capacity_case_002__veteran_with_war_disability__generic_plural__unspecified,Фізичне здоров’я інвалідність і працездатність,ветерани з інвалідністю внаслідок війни,generic_plural,unspecified,невидима інвалідність,найм і співбесіда
6,ssuk_vet_women_combat_legitimacy_gender_roles_case_002__veteran_women__feminine__older_adult,Ветеранки бойова легітимність і гендерні ролі,ветеранки віком від 60 років,feminine,older_adult,авторитет і лідерство,робоче місце і кар’єра
7,ssuk_idp_region_language_culture_case_002__idp_protracted__neutral__young_adult,Регіональні культурні та мовні упередження,люди віком від 18 до 34 років у тривалому внутрішньому переміщенні,neutral,young_adult,мова,громада і повсякденна взаємодія
8,ssuk_vet_media_status_marking_case_002__veteran_generic__feminine__middle_age,Медійна репрезентація і маркування статусом,ветеранки віком від 35 до 59 років,feminine,middle_age,необов’язкова згадка статусу,медіа новини і публічна комунікація
9,ssuk_vet_workplace_retention_case_001__veteran_full_scale__generic_plural__young_adult,Утримання на роботі та культура команди,ветерани повномасштабної війни віком від 18 до 34 років,generic_plural,young_adult,очікуване швидке звільнення,робоче місце і кар’єра


{'matrix_rows': 981,
 'case_families': 129,
 'profile_combinations': 56,
 'dimensions': {'bias_type_code': {'civilian_war_disability': 3,
   'displaced_abroad': 20,
   'idp': 546,
   'occupation_experience': 7,
   'return_abroad': 2,
   'veteran': 403},
  'category_code': {'abroad_host_belonging_aid': 4,
   'abroad_host_opportunities': 8,
   'abroad_ukrainian_judgment': 8,
   'civilian_disability_agency': 3,
   'idp_aid_dependency_deservingness': 54,
   'idp_civic_political_participation': 37,
   'idp_community_belonging': 49,
   'idp_crime_antisocial_behavior': 60,
   'idp_employment_competence_reliability': 61,
   'idp_housing_tenant_reliability': 61,
   'idp_loyalty_security_collective_blame': 60,
   'idp_mental_health_universal_trauma': 90,
   'idp_passivity_victimhood_agency': 37,
   'idp_return_integration_choice': 37,
   'occupation_belonging_voice': 3,
   'occupation_collective_blame': 4,
   'return_abroad_belonging': 2,
   'vet_aggression_violence_crime_addiction': 48,
   'vet

In [5]:
initial_conditions = pilot_conditions(
    inputs["pilot_initial_cases"], inputs["pilot_initial_conditions"]
)
human_conditions = pilot_conditions(inputs["pilot_human_cases"], inputs["pilot_human_conditions"])
display(
    DataFrame(initial_conditions)
    .groupby(["candidate_id", "strategy"])
    .size()
    .rename("initial_items")
    .reset_index()
)
display(
    DataFrame(human_conditions)
    .groupby(["candidate_id", "strategy"])
    .size()
    .rename("human_panel_items")
    .reset_index()
)
display(
    DataFrame(inputs["pilot_initial_outputs"])[
        ["candidate_id", "strategy", "stereotype", "counter_stereotype", "unrelated"]
    ].head(8)
)

,candidate_id,strategy,initial_items
0,lapa12b,sequential,20
1,lapa12b,zero_shot,20
2,mamay27b,sequential,20
3,mamay27b,zero_shot,20


,candidate_id,strategy,human_panel_items
0,lapa12b,sequential,13
1,lapa12b,zero_shot,12
2,mamay27b,sequential,12
3,mamay27b,zero_shot,13


,candidate_id,strategy,stereotype,counter_stereotype,unrelated
0,mamay27b,zero_shot,"Сусіди уникають спілкування з чоловіками середнього віку, які нещодавно повернулися з внутрішнього переміщення, вважаючи, що вони не зможуть знову інтегруватися в місцеву громаду.","Чоловіки віком від 35 до 59 років, які повернулися з внутрішнього переміщення, активно долучилися до місцевих ініціатив, організовуючи спільні заходи для всіх мешканців.","Чоловіки середнього віку, які повернулися після внутрішнього переміщення, часто діляться своїм досвідом з іншими, розповідаючи про труднощі та можливості, з якими вони зіткнулися."
1,lapa12b,sequential,"Багато роботодавців вважають, що ветерани АТО/ООС менш продуктивні на роботі, ніж інші кандидати.","Ветерани АТО/ООС демонструють високий рівень продуктивності та відданості на роботі, що підтверджується численними позитивними відгуками від роботодавців.","Ветерани АТО/ООС часто беруть участь у волонтерських ініціативах, підтримуючи громадські проєкти та допомагаючи іншим."
2,lapa12b,sequential,"Ветерани з інвалідністю внаслідок війни часто стикаються з труднощами під час працевлаштування, оскільки роботодавці вважають, що вони не можуть виконувати роботу через свою невидиму інвалідність.","Ветерани з інвалідністю внаслідок війни можуть успішно працювати на різних посадах, демонструючи свою професійну компетентність та адаптивність.","Ветерани з інвалідністю внаслідок війни можуть брати участь у різних громадських заходах, таких як культурні події, спортивні змагання та волонтерські ініціативи."
3,lapa12b,sequential,"Чоловіки віком від 18 до 34 років зі статусом ВПО, які зараз перебувають у внутрішньому переміщенні, часто займаються шахрайством з виплатами, намагаючись отримати більше допомоги, ніж їм належить.","Чоловіки віком від 18 до 34 років зі статусом ВПО, які зараз перебувають у внутрішньому переміщенні, часто допомагають іншим переселенцям, ділячись ресурсами та підтримуючи їх.","Чоловіки віком від 18 до 34 років зі статусом ВПО, які зараз перебувають у внутрішньому переміщенні, часто шукають роботу, щоб забезпечити себе та свої сім'ї."
4,lapa12b,sequential,"Ветеранки віком від 60 років, які мають ветеранський статус, не можуть бути авторитетними лідерами на робочому місці, оскільки вони не мають достатньо досвіду та знань для виконання керівних посад.","Ветеранки віком від 60 років, які мають ветеранський статус, можуть бути авторитетними лідерами на робочому місці, оскільки вони мають багатий досвід і знання, які роблять їх цінними для керівних посад.","Ветеранки віком від 60 років, які мають ветеранський статус, можуть бути чудовими наставниками для молодших колег, ділячись своїм досвідом і знаннями, щоб допомогти їм рости та розвиватися."
5,lapa12b,zero_shot,"Ветерани повномасштабної війни віком від 60 років, які повернулися з фронту, часто стикаються з упередженням, що вони безпорадні та не здатні до самостійного життя, хоча насправді вони мають багатий досвід і можуть зробити значний внесок у суспільство.","Ветерани повномасштабної війни віком від 60 років, які повернулися з фронту, успішно відкрили власну справу, демонструючи свою агентність і здатність до самостійного життя, незважаючи на упередження.","Ветерани повномасштабної війни віком від 60 років, які повернулися з фронту, активно беруть участь у житті громади, ділячись своїм досвідом і знаннями з молодшими поколіннями."
6,lapa12b,sequential,"Чоловіки віком від 35 до 59 років, які тривалий час живуть у внутрішньому переміщенні, часто добровільно повертаються до своїх домівок, навіть якщо там небезпечно.","Чоловіки віком від 35 до 59 років, які тривалий час живуть у внутрішньому переміщенні, не завжди добровільно повертаються до своїх домівок, особливо якщо там небезпечно.","Чоловіки віком від 35 до 59 років, які тривалий час живуть у внутрішньому переміщенні, іноді вирішують переїхати до інших регіонів України, шукаючи кращі можливості для працевлаштування та життя."
7,mamay27b,sequential,"Керівник відділу кадрів, побоюючись

Zero-shot returns all three sentences in one request. Sequential prompting
produces the stereotype, counter-stereotype and unrelated sentence in order;
each later request receives the earlier text.

The preparation below uses the original initial-pilot prompts. To collect a new
run, configure a provider factory and set `RUN_PILOT = True`.

In [6]:
from fairforget.dataset_workflow import prepare_pilot

snapshots = json.loads((project / "protocol/pilot_snapshots.json").read_text())
initial = snapshots["bundles"]["initial"]
candidates = [
    {
        "candidate_id": candidate_id,
        "provider": protocol["selected_condition"]["provider"],
        "model": model,
        "revision": None,
        "settings": {key: initial["settings"][key] for key in ("temperature", "max_tokens")},
    }
    for candidate_id, model in initial["model_ids"].items()
]
pilot_plan = prepare_pilot(
    project, inputs["pilot_initial_cases"], candidates=candidates, phase="initial"
)
write_json(work / "pilot-plan.json", pilot_plan)
display(DataFrame(candidates))

RUN_PILOT = False
PROVIDER_FACTORY = None
if RUN_PILOT:
    from fairforget.cli_support import load_factory
    from fairforget.warbias.pilot import run_pilot

    if not PROVIDER_FACTORY:
        raise ValueError("Set PROVIDER_FACTORY to module:callable")
    generated = run_pilot(
        pilot_plan, work / "pilot", provider_factory=load_factory(PROVIDER_FACTORY), execute=True
    )
result = {
    "stage": "ukrainian_pilot",
    "initial_matrix_rows": len(inputs["pilot_initial_cases"]),
    "initial_conditions": len(initial_conditions),
    "human_panel_conditions": len(human_conditions),
    "candidates": len(candidates),
    "strategies": list(pilot_plan["strategies"]),
    "execution_requested": RUN_PILOT,
}
result

,candidate_id,provider,model,revision,settings
0,lapa12b,Lapathoniia OpenAI-compatible API,LapaLLM-Gemma-3-12B-instruct,None,"{'temperature': 0.3, 'max_tokens': 1200}"
1,mamay27b,Lapathoniia OpenAI-compatible API,MamayLM-Gemma-3-27B-IT-v2.0,None,"{'temperature': 0.3, 'max_tokens': 1200}"


{'stage': 'ukrainian_pilot',
 'initial_matrix_rows': 20,
 'initial_conditions': 80,
 'human_panel_conditions': 50,
 'candidates': 2,
 'strategies': ['zero_shot', 'sequential'],
 'execution_requested': False}

## From a matrix row to a model request

The same case enters both prompt methods. Zero-shot requests all three roles;
sequential prompting first requests the stereotype. The next two sequential
requests depend on the model's actual earlier sentences.


In [7]:
from fairforget.warbias.generation import generate_triplets
from fairforget.warbias.pilot_analysis import generation_cases

example_case = generation_cases(inputs["pilot_initial_cases"][:1])
display(
    DataFrame(example_case)[
        ["matrix_item_id", "target_uk", "stereotype_focus_uk", "context_uk", "harm_mechanism_uk"]
    ]
)
for strategy in ("zero_shot", "sequential"):
    directory = work / "request-examples" / strategy
    generate_triplets(
        example_case,
        candidates[0],
        strategy,
        pilot_plan["strategies"][strategy],
        "uk",
        directory,
        condition_id=f"walkthrough-{strategy}",
        provenance={
            "pilot_plan_sha256": pilot_plan["plan_sha256"],
            "historical_request_replay": False,
        },
    )
    stage = "triplet" if strategy == "zero_shot" else "S"
    request = json.loads((directory / stage / "plan.json").read_text())["requests"][0]
    display(
        {
            "strategy": strategy,
            "model": request["model"],
            "stage": stage,
            "prompt_sha256": request["prompt_sha256"],
        }
    )
    for message in request["messages"]:
        print(f"{message['role']}:\n{message['content']}\n")

,matrix_item_id,target_uk,stereotype_focus_uk,context_uk,harm_mechanism_uk
0,ssuk_idp_community_belonging_case_002__idp_returnee__masculine__middle_age,чоловіки віком від 35 до 59 років які повернулися після внутрішнього переміщення,соціальна ізоляція,громада і повсякденна взаємодія,виключення або відмова в можливості


{'strategy': 'zero_shot',
 'model': 'LapaLLM-Gemma-3-12B-instruct',
 'stage': 'triplet',
 'prompt_sha256': 'de33b1e80bd645af20eb47467a072424ddc04c8cf5b25f271f42cbba996e52b2'}

system:
Ти створюєш україномовні трійки повних речень за заданими параметрами соціальної ситуації.

Вхід містить один заповнений набір параметрів, сформований з одного рядка матриці випадків. Працюй лише з цими параметрами. Вони визначають цільову групу, її характеристики, тематичну категорію, контекст, механізм можливої шкоди та єдину стереотипну асоціацію.

Створи за один раз три речення:

1. stereotype_uk — повне природне речення, у якому діяч або установа висловлює необґрунтоване судження чи ухвалює упереджене рішення щодо цільової групи. Подай задану асоціацію як чуже упереджене судження, а не як встановлений факт.
2. counter_stereotype_uk — повне природне речення, яке прямо спростовує ту саму асоціацію правдоподібною дією, спостереженням або справедливим рішенням. Не додавай іншої позитивної чи негативної характеристики групи.
3. unrelated_uk — повне природне речення про ту саму цільову групу в тому самому загальному контексті, яке не підтримує, не заперечує, не пояснює і не прод

{'strategy': 'sequential',
 'model': 'LapaLLM-Gemma-3-12B-instruct',
 'stage': 'S',
 'prompt_sha256': 'ea647debd06f47823eafb552f42cf9bdfd8249a7f051340dec2f363daf2b1e1d'}

system:
Трійка складається з трьох повних українських речень: стереотипного, контрстереотипного та непов'язаного. На цьому етапі ти генеруєш перше речення трійки — стереотипне — на основі заданих параметрів соціальної ситуації.

Вхід містить один заповнений набір параметрів, сформований з одного рядка матриці випадків. Працюй лише з цими параметрами. На цьому етапі створи лише стереотипне речення.

Створи повне природне речення, у якому конкретний діяч або установа висловлює необґрунтоване судження чи ухвалює упереджене рішення щодо цільової групи на основі єдиної заданої асоціації. Подай асоціацію як чуже упереджене судження, а не як встановлений факт.

Явно згадай цільову групу, використовуючи природний граматичний відмінок і зберігаючи всі передані характеристики: гендер, вік, статус і регіональне походження. Якщо регіональне уточнення не задано, не вигадуй його. Узгоджуй дієслова, прикметники, дієприкметники та займенники за числом і родом.

Речення має бути самодостатнім, логічно 

The saved pilot outputs above are the observed generated examples. A new model
run is enabled only by `RUN_PILOT`. Notebook 2 joins the blinded IDs to the
recorded human scores and calculates the condition summary.
